In [ ]:
!pip install pyspark -q
from google.colab import drive
drive.mount('/content/drive')

import os, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

BASE_PATH = "/content/drive/MyDrive/HM-DATA/"
INPUT_TRANS = BASE_PATH + "processed_v2/cleaned_transactions.parquet"
OUTPUT_DIR = BASE_PATH + "outputs_v2/candidates/"

spark = SparkSession.builder \
    .appName("Retrieval_Popularity") \
    .config("spark.driver.memory", "10g") \
    .getOrCreate()

Mounted at /content/drive


In [ ]:
transactions = spark.read.parquet(INPUT_TRANS)
max_date = transactions.select(F.max("t_dat_date")).collect()[0][0]

test_start = max_date - datetime.timedelta(days=7)
val_start = test_start - datetime.timedelta(days=7)
train_hist_start = val_start - datetime.timedelta(days=42)
test_hist_start = test_start - datetime.timedelta(days=42)

train_hist_df = transactions.filter((F.col("t_dat_date") >= train_hist_start) & (F.col("t_dat_date") < val_start))
test_hist_df = transactions.filter((F.col("t_dat_date") >= test_hist_start) & (F.col("t_dat_date") < test_start))

In [ ]:
def generate_popularity_candidates(history_df, top_n=30):
    hist_max_date = history_df.select(F.max("t_dat_date")).collect()[0][0]
    recent_week_start = hist_max_date - datetime.timedelta(days=7)

    top_items = history_df.filter(F.col("t_dat_date") >= recent_week_start) \
        .groupBy("article_id").count().orderBy(F.col("count").desc()).limit(top_n).select("article_id")

    unique_users = history_df.select("customer_id").dropDuplicates()
    candidates = unique_users.crossJoin(F.broadcast(top_items)).withColumn("strategy", F.lit("popularity"))
    return candidates

def evaluate_recall(candidates_df, target_df, target_start, target_end):
    actuals = target_df.filter((F.col("t_dat_date") >= target_start) & (F.col("t_dat_date") < target_end)) \
        .select("customer_id", "article_id").dropDuplicates()

    total_actuals = actuals.count()
    hits = actuals.join(candidates_df, ["customer_id", "article_id"], "inner").dropDuplicates().count()
    recall = hits / total_actuals if total_actuals > 0 else 0

    print(f"Actuals: {total_actuals} | Hits: {hits} | Recall: {recall:.4f}")

In [ ]:
train_cands = generate_popularity_candidates(train_hist_df, top_n=30)
train_cands.write.mode("overwrite").parquet(OUTPUT_DIR + "train_popularity.parquet")

test_cands = generate_popularity_candidates(test_hist_df, top_n=30)
test_cands.write.mode("overwrite").parquet(OUTPUT_DIR + "test_popularity.parquet")

print("Evaluating TEST set:")
evaluate_recall(test_cands, transactions, test_start, max_date)

Evaluating TEST set:
Actuals: 207996 | Hits: 5081 | Recall: 0.0244
